In [1]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
shl_intern_hiring_assessment_2025_path = kagglehub.competition_download('shl-intern-hiring-assessment-2025')
print('Data source import complete.')

100%|██████████| 1.23G/1.23G [00:58<00:00, 22.7MB/s]

Extracting files...


Data source import complete.


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from tqdm.auto import tqdm
import warnings
import logging
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
from sklearn.model_selection import KFold
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

In [4]:
DATA_DIR = shl_intern_hiring_assessment_2025_path
TRAIN_CSV_PATH = os.path.join(DATA_DIR, "dataset/csvs/train.csv")
TEST_CSV_PATH = os.path.join(DATA_DIR, "dataset/csvs/test.csv")
TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "dataset/audios/train")
TEST_AUDIO_DIR = os.path.join(DATA_DIR, "dataset/audios/test")
SUBMISSION_PATH = "./submission.csv"
PREPROCESSED_TRAIN_CSV = "./train_with_transcripts.csv"
WHISPER_MODEL = "openai/whisper-base"
BERT_MODEL = "roberta-base"
MODEL_SAVE_PATH = "./fine-tuned-grammar-scorer"
BERT_MAX_LENGTH = 512
SCORE_MIN = 0.0
SCORE_MAX = 5.0

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)
print(f"Loaded train.csv: {len(train_df)} samples")
print(f"Loaded test.csv: {len(test_df)} samples")
print(train_df.head(3))

Loaded train.csv: 409 samples
Loaded test.csv: 197 samples
    filename  label
0  audio_173    3.0
1  audio_138    3.0
2  audio_127    2.0


In [7]:
whisper_pipeline = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    device=0 if DEVICE == "cuda" else -1,
)
print("Whisper pipeline loaded successfully.")

def get_transcription(audio_file_path: str) -> str:
    if not os.path.exists(audio_file_path):
        print(f"Warning: Audio file not found at {audio_file_path}")
        return "AUDIO_FILE_NOT_FOUND"
    try:
        audio_array, sampling_rate = librosa.load(audio_file_path, sr=16000, mono=True)
        result = whisper_pipeline(
            {"raw": audio_array, "sampling_rate": sampling_rate},
            chunk_length_s=30,
            batch_size=8
        )
        return result["text"].strip()
    except Exception as e:
        print(f"Error processing {audio_file_path}: {e}")
        return "TRANSCRIPTION_ERROR"

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Whisper pipeline loaded successfully.


In [8]:
if not os.path.exists(PREPROCESSED_TRAIN_CSV):
    print(f"'{PREPROCESSED_TRAIN_CSV}' not found. Starting transcription process...")
    transcripts = []
    for filename in tqdm(train_df['filename'], desc="Transcribing Training Audio"):
        clean_filename = f"{filename.strip()}.wav"
        file_path_train = os.path.join(TRAIN_AUDIO_DIR, clean_filename)
        file_path_test = os.path.join(TEST_AUDIO_DIR, clean_filename)
        final_path = None

        if os.path.exists(file_path_train):
            final_path = file_path_train
        elif os.path.exists(file_path_test):
            final_path = file_path_test
        if final_path:
            transcript = get_transcription(final_path)
        else:
            transcript = "AUDIO_FILE_NOT_FOUND"

        transcripts.append(transcript)
    train_df['transcription'] = transcripts
    train_df.to_csv(PREPROCESSED_TRAIN_CSV, index=False)
    print(f"Transcription complete. Data saved to '{PREPROCESSED_TRAIN_CSV}'")
else:
    print(f"Found existing '{PREPROCESSED_TRAIN_CSV}'. Loading directly.")
    print("To re-run transcription, please delete this file and run this cell again.")
    train_df = pd.read_csv(PREPROCESSED_TRAIN_CSV)
print("\nPre-processed data:")
print(train_df.head())

'./train_with_transcripts.csv' not found. Starting transcription process...


Transcribing Training Audio:   0%|          | 0/409 [00:00<?, ?it/s]

Transcription complete. Data saved to './train_with_transcripts.csv'

Pre-processed data:
    filename  label                                      transcription
0  audio_173    3.0  My favorite place to visit will be Japan, beca...
1  audio_138    3.0  I love to reading on my hobbies as reading. Em...
2  audio_127    2.0  My favorite place to visit is the Malayites ne...
3   audio_95    2.0  I am going to tell about my hobby. And my hobb...
4   audio_73    3.5  This is a tough one, so my bestie of my life i...


In [9]:
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
def preprocess_for_bert(examples):
    tokenized = tokenizer(
        examples["transcription"],
        truncation=True,
        padding="max_length",
        max_length=BERT_MAX_LENGTH
    )
    tokenized["labels"] = [float(label) for label in examples["label"]]
    return tokenized

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    pearson, _ = pearsonr(labels, predictions)
    return {"rmse": rmse, "pearsonr": pearson}

In [11]:
class RegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = torch.nn.MSELoss()
        loss = loss_fct(logits.squeeze(), labels.squeeze())
        return (loss, outputs) if return_outputs else loss

In [12]:
full_train_df = pd.read_csv(PREPROCESSED_TRAIN_CSV)
full_train_df['label'] = full_train_df['label'].astype(float)
original_rows = len(full_train_df)
full_train_df = full_train_df.dropna(subset=['transcription'])
full_train_df = full_train_df[~full_train_df['transcription'].str.contains("AUDIO_FILE_NOT_FOUND|TRANSCRIPTION_ERROR")]
cleaned_rows = len(full_train_df)
print(f"Loaded {cleaned_rows} valid training samples.")
full_dataset = Dataset.from_pandas(full_train_df)
print("Tokenizing full dataset...")
tokenized_full_dataset = full_dataset.map(preprocess_for_bert, batched=True)
final_test_df = pd.read_csv(TEST_CSV_PATH)
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
all_validation_metrics = []
all_test_predictions = []

Loaded 409 valid training samples.
Tokenizing full dataset...


Map:   0%|          | 0/409 [00:00<?, ? examples/s]

In [13]:
full_train_df = pd.read_csv(PREPROCESSED_TRAIN_CSV)
full_train_df['label'] = full_train_df['label'].astype(float)

original_rows = len(full_train_df)
full_train_df = full_train_df.dropna(subset=['transcription'])
full_train_df = full_train_df[~full_train_df['transcription'].str.contains("AUDIO_FILE_NOT_FOUND|TRANSCRIPTION_ERROR")]
cleaned_rows = len(full_train_df)
print(f"Loaded {cleaned_rows} valid training samples.")

full_dataset = Dataset.from_pandas(full_train_df)
tokenized_full_dataset = full_dataset.map(preprocess_for_bert, batched=True)
final_test_df = pd.read_csv(TEST_CSV_PATH)

N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
all_validation_metrics = []
all_test_predictions = []

for fold, (train_index, val_index) in enumerate(kf.split(tokenized_full_dataset)):
    print("\n" + "="*30)
    print(f"--- STARTING FOLD {fold + 1}/{N_SPLITS} ---")
    print("="*30)

    train_dataset = tokenized_full_dataset.select(train_index)
    eval_dataset = tokenized_full_dataset.select(val_index)

    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL,
        num_labels=1
    ).to(DEVICE)

    training_args = TrainingArguments(
        output_dir=f"./bert_checkpoints_fold_{fold+1}",
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=50,
        weight_decay=0.01,
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="rmse",
        greater_is_better=False,
        report_to="none",
        label_names=["labels"]
    )

    trainer = RegressionTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer,
    )

    trainer.train()

    metrics = trainer.evaluate()
    all_validation_metrics.append(metrics)
    print(f"Fold {fold+1} Metrics: RMSE={metrics['eval_rmse']:.4f}, Pearson={metrics['eval_pearsonr']:.4f}")

    inference_model = AutoModelForSequenceClassification.from_pretrained(trainer.state.best_model_checkpoint).to(DEVICE)
    inference_tokenizer = AutoTokenizer.from_pretrained(trainer.state.best_model_checkpoint)
    inference_model.eval()

    fold_test_predictions = []

    for filename in tqdm(final_test_df['filename'], desc=f"Scoring Test Audio (Fold {fold+1})"):
        clean_filename = f"{filename.strip()}.wav"
        file_path_train = os.path.join(TRAIN_AUDIO_DIR, clean_filename)
        file_path_test = os.path.join(TEST_AUDIO_DIR, clean_filename)

        final_path = None
        if os.path.exists(file_path_train):
            final_path = file_path_train
        elif os.path.exists(file_path_test):
            final_path = file_path_test

        if final_path:
            transcription = get_transcription(final_path)
        else:
            transcription = "AUDIO_FILE_NOT_FOUND"

        if transcription in ["AUDIO_FILE_NOT_FOUND", "TRANSCRIPTION_ERROR"]:
            predicted_score = full_train_df['label'].mean()
        else:
            all_tokens = inference_tokenizer.tokenize(transcription)
            if len(all_tokens) <= (BERT_MAX_LENGTH - 2):
                inputs = inference_tokenizer(transcription, return_tensors="pt", truncation=True, padding="max_length", max_length=BERT_MAX_LENGTH).to(DEVICE)
                with torch.no_grad():
                    predicted_score = inference_model(**inputs).logits[0].item()
            else:
                chunk_size = BERT_MAX_LENGTH - 2
                stride = 410
                chunks = []
                for i in range(0, len(all_tokens), stride):
                    chunks.append(all_tokens[i : i + chunk_size])

                chunk_scores = []
                for token_chunk in chunks:
                    chunk_text = inference_tokenizer.convert_tokens_to_string(token_chunk)
                    inputs = inference_tokenizer(chunk_text, return_tensors="pt", truncation=True, padding="max_length", max_length=BERT_MAX_LENGTH).to(DEVICE)
                    with torch.no_grad():
                        chunk_scores.append(inference_model(**inputs).logits[0].item())
                predicted_score = np.mean(chunk_scores)

        clamped_score = max(SCORE_MIN, min(SCORE_MAX, predicted_score))
        fold_test_predictions.append(clamped_score)

    all_test_predictions.append(fold_test_predictions)
    print(f"--- FOLD {fold + 1} COMPLETE ---")

print("\n" + "="*30)
print("--- ALL FOLDS COMPLETE ---")
print("="*30)

print("Final Validation Metrics (Average across all folds):")
avg_rmse = np.mean([m['eval_rmse'] for m in all_validation_metrics])
avg_pearson = np.mean([m['eval_pearsonr'] for m in all_validation_metrics])
print(f"Average Validation RMSE: {avg_rmse:.4f}")
print(f"Average Validation Pearson: {avg_pearson:.4f}")

print("Averaging test predictions from all 5 models...")
predictions_array = np.array(all_test_predictions)
final_avg_predictions = np.mean(predictions_array, axis=0)

submission_df = pd.DataFrame({
    "filename": final_test_df['filename'],
    "label": final_avg_predictions
})

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"\nSubmission file created at {SUBMISSION_PATH}")
print(submission_df.head())

Loaded 409 valid training samples.


Map:   0%|          | 0/409 [00:00<?, ? examples/s]


--- STARTING FOLD 1/5 ---


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

{'loss': 3.7994, 'grad_norm': 16.09579849243164, 'learning_rate': 4e-05, 'epoch': 1.0}
{'eval_loss': 0.6014124155044556, 'eval_rmse': 0.7755078051830367, 'eval_pearsonr': -0.07741987705230713, 'eval_runtime': 2.1473, 'eval_samples_per_second': 38.188, 'eval_steps_per_second': 5.123, 'epoch': 1.0}
{'loss': 0.6674, 'grad_norm': 25.294025421142578, 'learning_rate': 4e-05, 'epoch': 2.0}
{'eval_loss': 0.6088805794715881, 'eval_rmse': 0.7803080029524163, 'eval_pearsonr': 0.22745883464813232, 'eval_runtime': 2.1918, 'eval_samples_per_second': 37.412, 'eval_steps_per_second': 5.019, 'epoch': 2.0}
{'loss': 0.6655, 'grad_norm': 18.310714721679688, 'learning_rate': 2.67741935483871e-05, 'epoch': 3.0}
{'eval_loss': 0.5988094806671143, 'eval_rmse': 0.7738278107351236, 'eval_pearsonr': 0.48367995023727417, 'eval_runtime': 2.1558, 'eval_samples_per_second': 38.037, 'eval_steps_per_second': 5.103, 'epoch': 3.0}
{'loss': 0.4556, 'grad_norm': 12.077299118041992, 'learning_rate': 1.3548387096774195e-05, 

...

--- FOLD 3 COMPLETE ---

--- STARTING FOLD 4/5 ---
{'loss': 3.7985, 'grad_norm': 44.06706237792969, 'learning_rate': 4e-05, 'epoch': 1.0}
{'eval_loss': 0.4650481641292572, 'eval_rmse': 0.6819444214388586, 'eval_pearsonr': 0.24615097045898438, 'eval_runtime': 2.1159, 'eval_samples_per_second': 38.753, 'eval_steps_per_second': 5.199, 'epoch': 1.0}
{'loss': 0.6345, 'grad_norm': 16.162691116333008, 'learning_rate': 4e-05, 'epoch': 2.0}
{'eval_loss': 0.3541308045387268, 'eval_rmse': 0.595088904735021, 'eval_pearsonr': 0.5298246741294861, 'eval_runtime': 2.1458, 'eval_samples_per_second': 38.213, 'eval_steps_per_second': 5.126, 'epoch': 2.0}
{'loss': 0.3639, 'grad_norm': 5.810674667358398, 'learning_rate': 2.67741935483871e-05, 'epoch': 3.0}
{'eval_loss': 0.4041666090488434, 'eval_rmse': 0.6357409686708266, 'eval_pearsonr': 0.5519742369651794, 'eval_runtime': 2.1319, 'eval_samples_per_second': 38.464, 'eval_steps_per_second': 5.16, 'epoch': 3.0}
{'loss': 0.2485, 'grad_norm': 29.7793636322021

Scoring Test Audio (Fold 4):   0%|          | 0/197 [00:00<?, ?it/s]

--- FOLD 4 COMPLETE ---

--- STARTING FOLD 5/5 ---
{'loss': 3.7458, 'grad_norm': 22.164011001586914, 'learning_rate': 4e-05, 'epoch': 1.0}
{'eval_loss': 0.8091274499893188, 'eval_rmse': 0.8995151193778339, 'eval_pearsonr': 0.33437129855155945, 'eval_runtime': 2.1069, 'eval_samples_per_second': 38.444, 'eval_steps_per_second': 5.221, 'epoch': 1.0}
{'loss': 0.746, 'grad_norm': 22.142681121826172, 'learning_rate': 4e-05, 'epoch': 2.0}
{'eval_loss': 0.6238781809806824, 'eval_rmse': 0.7898596334699775, 'eval_pearsonr': 0.3532603085041046, 'eval_runtime': 2.0611, 'eval_samples_per_second': 39.3, 'eval_steps_per_second': 5.337, 'epoch': 2.0}
{'loss': 0.586, 'grad_norm': 22.270620346069336, 'learning_rate': 2.67741935483871e-05, 'epoch': 3.0}
{'eval_loss': 0.5133343935012817, 'eval_rmse': 0.7164736234544343, 'eval_pearsonr': 0.4433876574039459, 'eval_runtime': 2.0834, 'eval_samples_per_second': 38.879, 'eval_steps_per_second': 5.28, 'epoch': 3.0}
{'loss': 0.4314, 'grad_norm': 7.669543266296387

Scoring Test Audio (Fold 5):   0%|          | 0/197 [00:00<?, ?it/s]

--- FOLD 5 COMPLETE ---

--- ALL FOLDS COMPLETE ---
Final Validation Metrics (Average across all folds):
Average Validation RMSE: 0.6082
Average Validation Pearson: 0.6432
Averaging test predictions from all 5 models...

Submission file created at ./submission.csv
    filename     label
0  audio_141  2.619376
1  audio_114  2.578414
2   audio_17  2.881878
3   audio_76  3.565631
4  audio_156  2.399000
